In [ ]:
# Preprocessing before fused PET-CT generation
# Tumor-region enhancement for all PET images using corresponding masks
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# ============================================================
# 1. Define folders
# ============================================================
pet_folder = r".../PET-CT/PET"
mask_folder = r".../PET-CT/Mask"
output_folder = r".../PET-CT/Superimposed Tumor"

# Create output folder
os.makedirs(output_folder, exist_ok=True)

# ============================================================
# 2. Read all PET images
# ============================================================
pet_files = sorted(glob.glob(os.path.join(pet_folder, "*.png")))
print(f"Total PET images found: {len(pet_files)}")

# ============================================================
# 3. Process all PET images
# ============================================================
processed_count = 0
missing_mask_count = 0
failed_count = 0

for image_path in pet_files:
    # --------------------------------------------------------
    # Get PET filename
    # --------------------------------------------------------
    image_filename = os.path.basename(image_path)
    # Remove extension
    base_name = os.path.splitext(image_filename)[0]

    # --------------------------------------------------------
    # Find corresponding mask
    # --------------------------------------------------------
    mask_filename = base_name.replace(" - Copy", " -mask") + ".png"
    mask_path = os.path.join(mask_folder, mask_filename)

    # --------------------------------------------------------
    # Check mask availability
    # --------------------------------------------------------
    if not os.path.exists(mask_path):

        print(
            f"WARNING: Mask not found for: "
            f"{image_filename}"
        )

        missing_mask_count += 1
        continue


    # ========================================================
    # 4. Load PET image
    # ========================================================
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:

        print(
            f"WARNING: Could not read PET image: "
            f"{image_filename}"
        )

        failed_count += 1
        continue

    image = image.astype(np.float32)

    # ========================================================
    # 5. Load mask
    # ========================================================
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:

        print(
            f"WARNING: Could not read mask: "
            f"{mask_filename}"
        )

        failed_count += 1
        continue

    # ========================================================
    # 6. Resize mask to PET dimensions
    # ========================================================
    mask_resized = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

    # ========================================================
    # 7. Create smoothed copies
    # ========================================================
    smoothed_c = cv2.GaussianBlur(image, (5, 5), 0)
    smoothed_im = image.copy()

    # ========================================================
    # 8. Calculate maximum values
    # ========================================================
    max_value = np.max(smoothed_c)
    max_value1 = np.max(smoothed_im)

    # ========================================================
    # 9. Tumor enhancement
    # ========================================================
    # Tumor region:maximum value × 1.5
    # Non-tumor region:original value × 0.5
    # ========================================================
    tumor_region = mask_resized == 255
    smoothed_c[tumor_region] = max_value * 1.5
    smoothed_im[tumor_region] = max_value1 * 1.5
    smoothed_c[~tumor_region] *= 0.5
    smoothed_im[~tumor_region] *= 0.5

    # ========================================================
    # 10. Clip values to [0, 255]
    # ========================================================
    smoothed_c = np.clip(smoothed_c, 0, 255).astype(np.uint8)
    smoothed_im = np.clip(smoothed_im, 0, 255).astype(np.uint8)

    # ========================================================
    # 11. Save output
    # ========================================================
    # Here we save the Gaussian-smoothed version.
    # ========================================================

    output_filename = image_filename
    output_path = os.path.join(output_folder, output_filename)
    cv2.imwrite(output_path, smoothed_c)

    processed_count += 1

    print(
        f"[{processed_count}/{len(pet_files)}] "
        f"Processed: {image_filename}"
    )

# ============================================================
# 12. Processing summary
# ============================================================
print("\n==============================================")
print(f"PET images found     : {len(pet_files)}")
print(f"Images processed     : {processed_count}")
print(f"Missing masks        : {missing_mask_count}")
print(f"Failed images        : {failed_count}")
print(f"Output folder        : {output_folder}")
print("==============================================")

# ============================================================
# 13. Display an example
# ============================================================
if processed_count > 0:

    # Find first valid PET-mask pair
    for image_path in pet_files:

        image_filename = os.path.basename(image_path)

        base_name = os.path.splitext(
            image_filename
        )[0]

        mask_filename = base_name.replace(" - Copy"," -mask") + ".png"

        mask_path = os.path.join(mask_folder, mask_filename)

        if os.path.exists(mask_path):
            break


    # Load original PET
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    # Load mask
    mask = cv2.imread(mask_path,cv2.IMREAD_GRAYSCALE)
    # Resize mask
    mask_resized = cv2.resize(mask,(image.shape[1], image.shape[0]),interpolation=cv2.INTER_NEAREST)


    # Reproduce preprocessing for visualization
    image_float = image.astype(np.float32)
    smoothed_c = cv2.GaussianBlur(image_float,(5, 5),0)
    max_value = np.max(smoothed_c)
    tumor_region = mask_resized == 255
    smoothed_c[tumor_region] = max_value * 1.5
    smoothed_c[~tumor_region] *= 0.5
    smoothed_c = np.clip(smoothed_c,0,255).astype(np.uint8)

    # ========================================================
    # Display
    # ========================================================
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 4, 1)
    plt.title("Original PET")
    plt.imshow(image, cmap="gray")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.title("Resized Mask")
    plt.imshow(mask_resized, cmap="gray")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.title("Smoothed PET")
    plt.imshow(cv2.GaussianBlur(image, (5, 5), 0),cmap="gray")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.title("Superimposed Tumor PET")
    plt.imshow(smoothed_c, cmap="gray")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Fused PET-CT generation using HOT colormap
# ============================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import os
import glob

# ============================================================
# 1. Define folders
# ============================================================
ct_folder = r".../PET-CT/CT"
superimposed_folder = r".../PET-CT/Superimposed Tumor"
output_folder = r".../PET-CT/fused PET-CT"

os.makedirs(output_folder, exist_ok=True)

# ============================================================
# 2. Read all CT images
# ============================================================
ct_files = sorted(glob.glob(os.path.join(ct_folder, "*.png")))
print(f"Total CT images found: {len(ct_files)}")

# ============================================================
# 3. Initialize counters
# ============================================================
processed_count = 0
missing_pet_count = 0
failed_count = 0

# ============================================================
# 4. Process all CT images
# ============================================================
for ct_path in ct_files:

    ct_filename = os.path.basename(ct_path)
    base_name = os.path.splitext(ct_filename)[0]

    possible_names = [base_name + ".png", base_name.replace(" - Copy - Copy", " - Copy") + ".png", base_name.replace(" - Copy", "") + ".png"]

    superimposed_path = None

    for filename in possible_names:
        test_path = os.path.join(superimposed_folder, filename)
        if os.path.exists(test_path):
            superimposed_path = test_path
            break

    if superimposed_path is None:
        print(f"WARNING: Superimposed Tumor image not found for {ct_filename}")
        missing_pet_count += 1
        continue

    # ========================================================
    # 5. Load CT and Superimposed Tumor PET
    # ========================================================
    ct_image = cv2.imread(ct_path, cv2.IMREAD_COLOR)
    pet_image = cv2.imread(superimposed_path, cv2.IMREAD_COLOR)

    if ct_image is None:
        print(f"WARNING: Could not read CT image: {ct_filename}")
        failed_count += 1
        continue

    if pet_image is None:
        print(f"WARNING: Could not read Superimposed Tumor image: {os.path.basename(superimposed_path)}")
        failed_count += 1
        continue

    # ========================================================
    # 6. Resize PET to CT dimensions
    # ========================================================
    if ct_image.shape != pet_image.shape:
        pet_image = cv2.resize(pet_image, (ct_image.shape[1], ct_image.shape[0]), interpolation=cv2.INTER_LINEAR)

    # ========================================================
    # 7. Fuse CT and Superimposed Tumor PET
    # ========================================================
    alpha = 0.7
    blended_image = cv2.addWeighted(ct_image, alpha, pet_image, 1 - alpha, 0)

    # ========================================================
    # 8. Convert fused image to grayscale
    # ========================================================
    blended_gray = cv2.cvtColor(blended_image, cv2.COLOR_BGR2GRAY)

    # ========================================================
    # 9. Normalize image
    # ========================================================
    norm_blended = blended_gray.astype(np.float32) / 255.0

    # ========================================================
    # 10. Apply HOT colormap
    # ========================================================
    hot_colormap = cm.get_cmap("hot")
    fused_hot = hot_colormap(norm_blended)[:, :, :3]

    # ========================================================
    # 11. Convert to 8-bit RGB
    # ========================================================
    fused_hot_8bit = (fused_hot * 255).astype(np.uint8)

    # ========================================================
    # 12. Convert RGB to BGR
    # ========================================================
    fused_hot_bgr = cv2.cvtColor(fused_hot_8bit, cv2.COLOR_RGB2BGR)

    # ========================================================
    # 13. Save fused PET-CT image
    # ========================================================
    output_filename = base_name + "-Fused_HOT.png"
    output_path = os.path.join(output_folder, output_filename)
    cv2.imwrite(output_path, fused_hot_bgr)

    processed_count += 1
    print(f"[{processed_count}/{len(ct_files)}] Processed: {ct_filename}")

# ============================================================
# 14. Processing summary
# ============================================================
print("\n================================================")
print(f"CT images found       : {len(ct_files)}")
print(f"Images processed      : {processed_count}")
print(f"Missing PET images    : {missing_pet_count}")
print(f"Failed images         : {failed_count}")
print(f"Output folder         : {output_folder}")
print("================================================")

# ============================================================
# 15. Display first successfully processed pair
# ============================================================
if processed_count > 0:

    for ct_path in ct_files:

        ct_filename = os.path.basename(ct_path)
        base_name = os.path.splitext(ct_filename)[0]
        possible_names = [base_name + ".png", base_name.replace(" - Copy - Copy", " - Copy") + ".png", base_name.replace(" - Copy", "") + ".png"]
        superimposed_path = None

        for filename in possible_names:
            test_path = os.path.join(superimposed_folder, filename)
            if os.path.exists(test_path):
                superimposed_path = test_path
                break

        if superimposed_path is not None:
            break

    ct_image = cv2.imread(ct_path, cv2.IMREAD_COLOR)
    pet_image = cv2.imread(superimposed_path, cv2.IMREAD_COLOR)

    if ct_image.shape != pet_image.shape:
        pet_image = cv2.resize(pet_image, (ct_image.shape[1], ct_image.shape[0]), interpolation=cv2.INTER_LINEAR)

    blended_image = cv2.addWeighted(ct_image, 0.7, pet_image, 0.3, 0)
    blended_gray = cv2.cvtColor(blended_image, cv2.COLOR_BGR2GRAY)
    norm_blended = blended_gray.astype(np.float32) / 255.0
    hot_colormap = cm.get_cmap("hot")
    fused_hot = hot_colormap(norm_blended)[:, :, :3]
    fused_hot_8bit = (fused_hot * 255).astype(np.uint8)

    # ========================================================
    # 16. Display results
    # ========================================================
    plt.figure(figsize=(16, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(cv2.cvtColor(ct_image, cv2.COLOR_BGR2RGB))
    plt.title("CT Image")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(cv2.cvtColor(pet_image, cv2.COLOR_BGR2RGB))
    plt.title("Superimposed Tumor PET")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(blended_gray, cmap="gray")
    plt.title("Fused PET-CT (Gray)")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(fused_hot_8bit)
    plt.title("Fused PET-CT (HOT Colormap)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()